In [0]:
%sql
USE e_commerce.bronze;

In [0]:
%run "../src/utils/configuration"

In [0]:
%run "../src/utils/read"

In [0]:
%run "../src/utils/common_functions"

In [0]:
%run "../src/utils/rules"

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, input_file_name

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2025-10-25")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [0]:
order_item_schema = StructType(fields=[
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])


In [0]:
path = f"{land_folder_path}/{v_file_date}/olist_order_items_dataset_{v_file_date}.csv"

In [0]:
options = {
    "header": True,
    "sep": ","
}

In [0]:
df_order_item = read_data(path, format="csv", schema=order_item_schema, options=options)

In [0]:
reglas = []

In [0]:
if df_order_item.isEmpty():

    no_registros(df_order_item, "order_item_id", reglas, "order_item", v_file_date)

    df_reglas = spark.createDataFrame(reglas)
    df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

    df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
df_order_item_bronze = add_ingestion_timestamp(df_order_item)\
    .withColumn("environment", lit(v_environment))\
    .withColumn("file_date", lit(v_file_date))\
    .withColumn("source_name", input_file_name())

In [0]:
try:
    df_order_item_bronze.write.format("delta").option("mergeSchema", "true").mode("append").partitionBy("file_date").option("path", f"{bronze_folder_path}/order_item").saveAsTable("bronze.order_item")
except Exception as e:
    error_msg = handle_error(e, dbutils)
    print(error_msg)
    raise

In [0]:
%sql
SELECT file_date, COUNT(1) FROM bronze.order_item GROUP by file_date ORDER BY 1;

In [0]:
%sql
SELECT * FROM bronze.log_transformation;